In [ ]:
import re
import os
import logging
import time
from datetime import datetime

logger = logging.getLogger(__name__)
if not logger.handlers:
    handler = logging.FileHandler('log_lines.txt')
    formatter = logging.Formatter('%(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

def get_first_n_words(s, n):
    """Function to get the first n words from a given string s"""
    words = s.split()[:n]
    return ' '.join(words) if len(words) == n else '[short sentence]'

def get_logger(file_name):
    """Function to setup logger for each input file"""
    main_dir = '.' # replace with your main directory path if needed
    base_name = os.path.basename(file_name)
    name, _ = os.path.splitext(base_name)
    new_dir = os.path.join(main_dir, 'noBlanks', name + '_log')
    os.makedirs(new_dir, exist_ok=True)
    log_file_name = os.path.join(new_dir, name + '_log.txt')
    i = 1
    while os.path.isfile(log_file_name):
        log_file_name = os.path.join(new_dir, f'{name}_log-{i}.txt')
        i += 1
    logger = logging.getLogger(file_name)
    if not logger.handlers:
        handler = logging.FileHandler(log_file_name)
        formatter = logging.Formatter('%(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

def log_separator(logger):
    """Function to write separator line to the log file"""
    logger.info('-' * 80)

def log_header(logger, file_name):
    """Function to write header lines to the log file"""
    logger.info('=' * 80)
    logger.info(f'Time:\t{datetime.now().strftime("%H:%M:%S")}\t\tDate:\t{datetime.now().strftime("%Y-%m-%d")}')
    logger.info(f'Call:\tline_condenser()')
    logger.info(f'File:\t{file_name}')
    log_separator(logger)

def log_footer(logger, total_changes, process_time):
    """Function to write footer lines to the log file"""
    log_separator(logger)
    logger.info(f'Total changes made:\t{total_changes}')
    logger.info(f'Finished process in:\t{process_time * 1000:.3f} ms')
    log_separator(logger)

def get_output_file_name(file_name):
    """Function to generate the output file name"""
    main_dir = '.' # replace with your main directory path if needed
    base_name = os.path.basename(file_name)
    name, ext = os.path.splitext(base_name)
    new_dir = os.path.join(main_dir, 'noBlanks')
    os.makedirs(new_dir, exist_ok=True)
    output_file_name = os.path.join(new_dir, name + '_noBlanks' + ext)
    i = 1
    while os.path.isfile(output_file_name):
        output_file_name = os.path.join(new_dir, f'{name}_noBlanks-{i}{ext}')
        i += 1
    return output_file_name

def line_condenser(file_name):
    """Main function to condense lines in the input file"""
    start_time = time.time()
    total_changes = 0

    individual_logger = get_logger(file_name)
    log_header(logger, file_name)
    log_header(individual_logger, file_name)

    with open(file_name, 'r') as f:
        lines = f.readlines()

    output_lines = []
    last_speaker = None

    for i, line in enumerate(lines):
        line = line.rstrip() # remove trailing whitespaces/newlines

        # If the line is empty, skip it
        if line == '':
            total_changes += 1
            continue

        # If the line starts with a speaker's name
        if re.match(r'^[A-Z]+:', line):
            output_lines.append(line)
            last_speaker = line.split(':')[0]
        # If the line is a continuation of the last speaker's text
        elif last_speaker:
            output_lines[-1] += ' ' + line
            total_changes += 1
            individual_logger.info(f"Line {i+1} starting with: '{get_first_n_words(line, 3)}' moved to Line {len(output_lines)} starting with: '{get_first_n_words(output_lines[-1], 3)}'")

    # Writing the output file
    output_file_name = get_output_file_name(file_name)
    with open(output_file_name, 'w') as f:
        f.write('\n'.join(output_lines))

    log_footer(logger, total_changes, time.time() - start_time)
    log_footer(individual_logger, total_changes, time.time() - start_time)

    print(f'Output written to {output_file_name}')

file = "all_debates/2016/preclean/preclean-2016-09-26-debate.txt" 
line_condenser(file)


In [3]:
import os
import glob
import nltk
import json
import datetime
import re
import logging
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.probability import FreqDist

def get_most_common_words(text, num_words):
    words = nltk.word_tokenize(text)
    words = [word for word in words if word not in stopwords.words('english')]
    fdist = FreqDist(words)
    return fdist.most_common(num_words)

In [ ]:
def debate_stats(directory):
    nltk.download('punkt')
    nltk.download('stopwords')
    file_count = 0
    total_words = 0
    total_words_no_speaker_no_insertions = 0
    total_interruptions = 0
    total_speakers = set()
    speaker_stats = defaultdict(lambda: defaultdict(int))
    results = {}

    for file_name in glob.glob(f"{directory}/**/*.txt", recursive=True):
        with open(file_name, 'r', encoding='utf-8') as file:
            lines = file.readlines()
            unique_speakers = set()
            file_total_words = 0
            file_total_words_no_speaker_no_insertions = 0
            file_total_interruptions = 0
            speaker_turns = defaultdict(int)
            file_count += 1
            for line in lines:
                try:
                    speaker_name, text = line.split(':', 1)
                except ValueError:
                    continue
                unique_speakers.add(speaker_name)
                total_speakers.add(speaker_name)
                words = nltk.word_tokenize(text)
                file_total_words += len(words)
                total_words += len(words)
                speaker_turns[speaker_name] += 1
                speaker_stats[speaker_name]['turns'] = speaker_turns[speaker_name]
                speaker_stats[speaker_name]['words'] += len(words)
                speaker_stats[speaker_name]['sentences'] += text.count('.') + text.count('!') + text.count('?')
                speaker_stats[speaker_name]['interrupts'] += text.count('--')
                file_total_interruptions += text.count('--')
                total_interruptions += text.count('--')
                no_speaker_no_insertions = re.sub(r'\[.*?\]', '', text)
                no_speaker_no_insertions = re.sub(r'\(.*?\)', '', no_speaker_no_insertions)
                no_speaker_no_insertions_words = nltk.word_tokenize(no_speaker_no_insertions)
                file_total_words_no_speaker_no_insertions += len(no_speaker_no_insertions_words)
                total_words_no_speaker_no_insertions += len(no_speaker_no_insertions_words)

        results[file_name] = {
            "Unique speakers": len(unique_speakers),
            "Total words": file_total_words,
            "Total words (no speaker names or insertions)": file_total_words_no_speaker_no_insertions,
            "Total interruptions": file_total_interruptions,
            "Most common words": get_most_common_words(' '.join(lines), 20),
            "Speakers": {}
        }

    return results

In [ ]:
def writeitdown():
    import json
    import datetime
    results = debate_stats("all_debates")
    current_time = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
    with open(f"debate_stats-{current_time}.json", 'w') as f:
        json.dump(results, f, indent=4)

    return results

In [4]:
results = writeitdown()

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/pulpdesign/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pulpdesign/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
import json

# Read the existing JSON file
with open("debate_stats-2023-06-07-15-24-12.json", "r") as f:
    data = json.load(f)

# Write it back out with indentation
with open("debate_stats-2023-06-07-15-24-12.json", "w") as f:
    json.dump(data, f, indent=4)


In [ ]:
def scandir(directory):
    for file_name in glob.glob(f"{directory}/**/*.txt", recursive=True):
        with open(file_name, 'r', encoding='utf-8') as file:
            
            
def get_most_common_words(text, num_words):
    words = nltk.word_tokenize(text)
    words = [word for word in words if word not in stopwords.words('english')]
    fdist = FreqDist(words)
    return fdist.most_common(num_words)

In [8]:
import os

def get_size(start_path='.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

def print_files_and_sizes():
    cwd = os.getcwd()
    sizes_and_items = []

    for item in os.listdir(cwd):
        item_path = os.path.join(cwd, item)
        size_in_bytes = get_size(item_path)  # get total size of directory including all subdirectories and files

        if os.path.isfile(item_path):
            item_type = "File"
        else:
            item_type = "Directory"

        size_in_kb = size_in_bytes / 1024
        size_in_mb = size_in_kb / 1024
        size_in_gb = size_in_mb / 1024

        if size_in_gb >= 1:
            size = f"{size_in_gb:.2f} GB"
            size_in_bytes_compare = size_in_bytes / (1024**3)
        elif size_in_mb >= 1:
            size = f"{size_in_mb:.2f} MB"
            size_in_bytes_compare = size_in_bytes / (1024**2)
        else:
            size = f"{size_in_kb:.2f} KB"
            size_in_bytes_compare = size_in_bytes / 1024

        sizes_and_items.append((item, size, size_in_bytes_compare, item_type))

    sizes_and_items.sort(key=lambda x: x[2], reverse=True)  # sort by size in descending order

    for item, size, size_in_bytes_compare, item_type in sizes_and_items:
        print(f"{item_type}: {item}, Size: {size}")

print_files_and_sizes()


Directory: .old, Size: 19.50 MB
Directory: 2016 US Presidential Elections Tweets CSV, Size: 13.17 GB
Directory: all_debates, Size: 7.84 MB
File: Neo4j-b258737f-Created-2023-06-07.txt, Size: 0.00 KB
File: .DS_Store, Size: 0.00 KB
File: uspd-formatting-2.ipynb, Size: 0.00 KB
File: notes-overall, Size: 0.00 KB
File: scoreboard.ipynb, Size: 0.00 KB
File: notes-api.txt, Size: 0.00 KB
File: uspd-extraction-new.ipynb, Size: 0.00 KB
File: notes-uspdc, Size: 0.00 KB
File: uspd-formatting-3.ipynb, Size: 0.00 KB
File: uspet-1.ipynb, Size: 0.00 KB
File: log_scrape.txt, Size: 0.00 KB
File: debate_stats-2023-06-07-15-24-12.json, Size: 0.00 KB
File: uspd-inspection.ipynb, Size: 0.00 KB
File: uspd-formatting.ipynb, Size: 0.00 KB
File: notes-cleaning, Size: 0.00 KB
File: notes-eda, Size: 0.00 KB
